In [1]:
# Import core libraries and the shared scorecard functions from src/
import sys
sys.path.append('../src')

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from scorecard_utils import (
    calculate_woe_iv, calculate_woe_iv_categorical, collapse_rare_categories,
    merge_missing_into_nearest_bin, calculate_woe_iv_with_zero_bin,
    extract_edges, ordered_bin_woe, apply_frozen_woe
)

train_bureau = pd.read_csv('../data/processed/train_bureau.csv')
print(train_bureau.shape)

(307511, 156)


In [2]:
# Rebuild the exact same train/validation split used in 04_scorecard_woe.ipynb
# (same 17 final variables, same random_state), to reproduce the frozen
# pipeline from scratch as a validation of reproducibility.

FINAL_17_VARS = [
    'EXT_SOURCE_2', 'EXT_SOURCE_3', 'NAME_EDUCATION_TYPE_BINNED', 'EXT_SOURCE_1',
    'AMT_CREDIT', 'CODE_GENDER', 'YEARS_EMPLOYED', 'ORGANIZATION_TYPE_BINNED',
    'FLOORSMAX_AVG', 'BUREAU_CREDIT_ACTIVE_ACTIVE_COUNT', 'BUREAU_AMT_CREDIT_SUM_LIMIT_MEAN',
    'OCCUPATION_TYPE_BINNED', 'REGION_POPULATION_RELATIVE', 'DAYS_ID_PUBLISH',
    'BUREAU_AMT_CREDIT_SUM_DEBT_MEAN', 'DAYS_LAST_PHONE_CHANGE', 'BUREAU_DAYS_CREDIT_MEAN',
]

RAW_SOURCE = {
    'NAME_EDUCATION_TYPE_BINNED': 'NAME_EDUCATION_TYPE',
    'ORGANIZATION_TYPE_BINNED': 'ORGANIZATION_TYPE',
    'OCCUPATION_TYPE_BINNED': 'OCCUPATION_TYPE',
}

X = pd.DataFrame(index=train_bureau.index)
for var in FINAL_17_VARS:
    X[var] = train_bureau[RAW_SOURCE.get(var, var)]
X['SK_ID_CURR'] = train_bureau['SK_ID_CURR']
y = train_bureau['TARGET']

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
print("Train:", X_train.shape, "| Val:", X_val.shape)
print("TARGET rate (train):", y_train.mean().round(4), "| (val):", y_val.mean().round(4))

Train: (246008, 18) | Val: (61503, 18)
TARGET rate (train): 0.0807 | (val): 0.0807


In [3]:
# Check zero-concentration for BUREAU_AMT_CREDIT_SUM_DEBT_MEAN specifically,
# since it replaced the originally zero-corrected _SUM variant during
# correlation pruning and was never individually validated for this pattern.
n_missing = train_bureau['BUREAU_AMT_CREDIT_SUM_DEBT_MEAN'].isnull().sum()
n_zero = (train_bureau['BUREAU_AMT_CREDIT_SUM_DEBT_MEAN'] == 0).sum()
n_valid = train_bureau['BUREAU_AMT_CREDIT_SUM_DEBT_MEAN'].notnull().sum()
print(f"Missing: {n_missing} ({n_missing/len(train_bureau)*100:.1f}%)")
print(f"Zero (among non-null): {n_zero} ({n_zero/n_valid*100:.1f}%)")

Missing: 51380 (16.7%)
Zero (among non-null): 69689 (27.2%)


In [5]:
# Compare standard WoE vs. zero-isolated WoE for BUREAU_AMT_CREDIT_SUM_DEBT_MEAN,
# now confirmed to have 27.2% zero concentration among non-null values,
# a pattern similar to BUREAU_AMT_CREDIT_SUM_DEBT_SUM (Group 4, 29.2%),
# which received this treatment originally; this variable did not.

train_xy = X_train.copy()
train_xy['TARGET'] = y_train

standard_result, standard_iv = calculate_woe_iv(train_xy, 'BUREAU_AMT_CREDIT_SUM_DEBT_MEAN', min_bad_per_bin=0)
zero_result, zero_iv = calculate_woe_iv_with_zero_bin(train_xy, 'BUREAU_AMT_CREDIT_SUM_DEBT_MEAN', min_bad_per_bin=0)

print("=== Padrão (qcut simples) ===")
print(standard_result.round(4).to_string(index=False))
print(f"IV total: {standard_iv:.4f}\n")

print("=== Zero isolado ===")
print(zero_result.round(4).to_string(index=False))
print(f"IV total: {zero_iv:.4f}")

=== Padrão (qcut simples) ===
                       bin  n_total  n_bad  pct_good  pct_bad     woe  iv_component
(-1083614.6709999999, 0.0]    56647   3126    0.2367   0.1574  0.4079        0.0323
           (0.0, 4181.063]     4810    304    0.0199   0.0153  0.2624        0.0012
       (4181.063, 22444.5]    20486   1438    0.0842   0.0724  0.1511        0.0018
       (22444.5, 44196.06]    20485   1611    0.0835   0.0811  0.0284        0.0001
      (44196.06, 72305.55]    20486   1828    0.0825   0.0920 -0.1094        0.0010
     (72305.55, 112972.95]    20485   1883    0.0823   0.0948 -0.1421        0.0018
     (112972.95, 182056.5]    20486   2040    0.0816   0.1027 -0.2306        0.0049
    (182056.5, 348749.994]    20485   1999    0.0817   0.1007 -0.2081        0.0039
  (348749.994, 43650000.0]    20486   1673    0.0832   0.0842 -0.0126        0.0000
                   Missing    41152   3958    0.1645   0.1993 -0.1920        0.0067
IV total: 0.0537

=== Zero isolado ===
       

## Note: BUREAU_AMT_CREDIT_SUM_DEBT_MEAN zero-concentration check

This variable entered the final 17 via correlation pruning (replacing
BUREAU_AMT_CREDIT_SUM_DEBT_SUM, which did receive the Group 4 zero-isolation
treatment), and was never individually checked for the same pattern.
Confirmed 27.2% zero concentration among non-null values, similar to the
_SUM variant (29.2%). Tested zero-isolated WoE against the standard qcut
treatment: IV 0.0542 vs. 0.0537, a negligible difference. Kept the standard
treatment; the plain quantile binning already happened to isolate zero
values reasonably well in this specific variable's distribution.

In [6]:
# Freeze bin/category structures and WoE values for all 17 final variables,
# using X_train/y_train only. EXT_SOURCE_2 uses the Missing-merge treatment
# (Group 1); all others use plain calculate_woe_iv / calculate_woe_iv_categorical,
# confirmed sufficient for this variable set (BUREAU_AMT_CREDIT_SUM_DEBT_MEAN
# checked and confirmed not to need zero-isolation, see note above).

MISSING_MERGE_VARS = ['EXT_SOURCE_2']

train_xy = X_train.copy()
train_xy['TARGET'] = y_train

frozen = {}

for var in FINAL_17_VARS:

    if var in MISSING_MERGE_VARS:
        result, iv = merge_missing_into_nearest_bin(train_xy, var)
        edges = extract_edges(result)
        bin_woe_ordered = ordered_bin_woe(result)

        is_missing = train_xy[var].isnull()
        total_good = (train_xy['TARGET'] == 0).sum()
        total_bad = (train_xy['TARGET'] == 1).sum()
        n_bad_missing = train_xy.loc[is_missing, 'TARGET'].sum()
        n_good_missing = is_missing.sum() - n_bad_missing
        missing_woe = np.log(
            ((n_good_missing + 0.5) / (total_good + 0.5)) /
            ((n_bad_missing + 0.5) / (total_bad + 0.5))
        )
        nearest_woe = min(bin_woe_ordered, key=lambda w: abs(w - missing_woe))
        frozen[var] = {'type': 'numeric', 'edges': edges, 'bin_woe_ordered': bin_woe_ordered,
                        'zero_woe': None, 'missing_woe': nearest_woe}

    elif var in ['NAME_EDUCATION_TYPE_BINNED', 'ORGANIZATION_TYPE_BINNED', 'OCCUPATION_TYPE_BINNED']:
        raw_col = RAW_SOURCE[var]
        raw_series = X_train[var]
        collapsed_series = collapse_rare_categories(train_xy.assign(**{raw_col: raw_series}), raw_col)
        category_map = dict(zip(raw_series.astype('object').fillna('Missing'), collapsed_series))
        temp_df = pd.DataFrame({var: collapsed_series, 'TARGET': y_train})
        result, iv = calculate_woe_iv_categorical(temp_df, var, min_bad_per_bin=0)
        frozen[var] = {'type': 'categorical', 'table': result, 'category_map': category_map}

    elif pd.api.types.is_numeric_dtype(X_train[var]):
        result, iv = calculate_woe_iv(train_xy, var, min_bad_per_bin=0)
        edges = extract_edges(result)
        bin_woe_ordered = ordered_bin_woe(result)
        missing_rows = result[result['bin'] == 'Missing']
        frozen[var] = {'type': 'numeric', 'edges': edges, 'bin_woe_ordered': bin_woe_ordered,
                        'zero_woe': None,
                        'missing_woe': missing_rows['woe'].values[0] if len(missing_rows) else None}

    else:
        result, iv = calculate_woe_iv_categorical(train_xy, var, min_bad_per_bin=0)
        frozen[var] = {'type': 'categorical', 'table': result, 'category_map': None}

print(f"Frozen structures built for {len(frozen)} of {len(FINAL_17_VARS)} variables.")

AVISO [EXT_SOURCE_2]: 1 bin(s) com menos de 100 casos 'bad', WoE pode ser instável:
    bin  n_bad
Missing     41

'EXT_SOURCE_2': Missing (WoE=0.0375) fundido com bin (0.512, 0.566] (WoE original=0.0847)
'NAME_EDUCATION_TYPE': Other_grouped ainda instável, fundido com 'Higher education'
Frozen structures built for 17 of 17 variables.


In [7]:
# Transform X_train and X_val into WoE space using only the frozen
# structures above, pure lookup, nothing recalculated from X_val.

X_train_woe = pd.DataFrame(index=X_train.index)
X_val_woe = pd.DataFrame(index=X_val.index)

for var in FINAL_17_VARS:
    X_train_woe[var] = apply_frozen_woe(X_train[var], frozen[var])
    X_val_woe[var] = apply_frozen_woe(X_val[var], frozen[var])

print("X_train_woe:", X_train_woe.shape, "| X_val_woe:", X_val_woe.shape)
print("Nulls (train):", X_train_woe.isnull().sum().sum(), "| Nulls (val):", X_val_woe.isnull().sum().sum())

X_train_woe: (246008, 17) | X_val_woe: (61503, 17)
Nulls (train): 0 | Nulls (val): 0


In [8]:
X_train_woe.head(2)

,EXT_SOURCE_2,EXT_SOURCE_3,NAME_EDUCATION_TYPE_BINNED,EXT_SOURCE_1,AMT_CREDIT,CODE_GENDER,YEARS_EMPLOYED,ORGANIZATION_TYPE_BINNED,FLOORSMAX_AVG,BUREAU_CREDIT_ACTIVE_ACTIVE_COUNT,BUREAU_AMT_CREDIT_SUM_LIMIT_MEAN,OCCUPATION_TYPE_BINNED,REGION_POPULATION_RELATIVE,DAYS_ID_PUBLISH,BUREAU_AMT_CREDIT_SUM_DEBT_MEAN,DAYS_LAST_PHONE_CHANGE,BUREAU_DAYS_CREDIT_MEAN
181648,-0.449285,0.545394,0.433340,-0.049271,0.034827,0.154054,-0.260779,-0.073397,0.416358,0.087677,0.134208,-0.291701,-0.150727,0.281571,-0.109447,-0.156136,0.211081
229245,0.083924,-0.152144,-0.108462,-0.059184,0.196560,-0.250222,-0.351108,0.081116,-0.140620,-0.248603,-0.174112,-0.368124,-0.071707,-0.141896,-0.191957,-0.140761,-0.248488


In [9]:
# Final validation: does the rebuilt pipeline reproduce the numbers already
# published in the README and the technical presentation? This is the real
# test, not just "the code runs without error".

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve

# ... (célula de congelamento das 17 variáveis, usando as funções copiadas do 04)

model_check = LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced')
model_check.fit(X_train_woe, y_train)
auc_check = roc_auc_score(y_val, model_check.predict_proba(X_val_woe)[:, 1])

print("AUC reproduzido:", round(auc_check, 4), "| Esperado: 0.7406")
print("Bate?", abs(auc_check - 0.7406) < 0.0005)

AUC reproduzido: 0.7405 | Esperado: 0.7406
Bate? True


In [10]:
# Check where 'Academic degree' actually landed in the frozen category map
print(frozen['NAME_EDUCATION_TYPE_BINNED']['category_map'].get('Academic degree'))
print()
print(frozen['NAME_EDUCATION_TYPE_BINNED']['table'][['bin', 'woe']].round(4).to_string(index=False))

Higher education

                          bin     woe
              Lower secondary -0.3505
Secondary / secondary special -0.1085
            Incomplete higher -0.0728
             Higher education  0.4333


In [11]:
# Check DAYS_LAST_PHONE_CHANGE's missing_woe and bin count
print("missing_woe:", frozen['DAYS_LAST_PHONE_CHANGE']['missing_woe'])
print("n bins:", len(frozen['DAYS_LAST_PHONE_CHANGE']['bin_woe_ordered']))

missing_woe: -1.3336400834329205
n bins: 9


In [12]:
# Fix 1: DAYS_LAST_PHONE_CHANGE belongs to Group 1 (missing-merge), not the generic branch
MISSING_MERGE_VARS = ['EXT_SOURCE_2', 'DAYS_LAST_PHONE_CHANGE']

# Fix 2: NAME_EDUCATION_TYPE_BINNED had a specific single-category merge
# (Academic degree -> Higher education), not the generic rare-category collapse
SINGLE_MERGE_VARS = {'NAME_EDUCATION_TYPE_BINNED': {'Academic degree': 'Higher education'}}

In [13]:
# Freeze bin/category structures and WoE values for all 17 final variables,
# using X_train/y_train only. EXT_SOURCE_2 uses the Missing-merge treatment
# (Group 1); all others use plain calculate_woe_iv / calculate_woe_iv_categorical,
# confirmed sufficient for this variable set (BUREAU_AMT_CREDIT_SUM_DEBT_MEAN
# checked and confirmed not to need zero-isolation, see note above).

MISSING_MERGE_VARS = ['EXT_SOURCE_2', 'DAYS_LAST_PHONE_CHANGE']
SINGLE_MERGE_VARS = {'NAME_EDUCATION_TYPE_BINNED': {'Academic degree': 'Higher education'}}

train_xy = X_train.copy()
train_xy['TARGET'] = y_train

frozen = {}

for var in FINAL_17_VARS:

    if var in MISSING_MERGE_VARS:
        result, iv = merge_missing_into_nearest_bin(train_xy, var)
        edges = extract_edges(result)
        bin_woe_ordered = ordered_bin_woe(result)

        is_missing = train_xy[var].isnull()
        total_good = (train_xy['TARGET'] == 0).sum()
        total_bad = (train_xy['TARGET'] == 1).sum()
        n_bad_missing = train_xy.loc[is_missing, 'TARGET'].sum()
        n_good_missing = is_missing.sum() - n_bad_missing
        missing_woe = np.log(
            ((n_good_missing + 0.5) / (total_good + 0.5)) /
            ((n_bad_missing + 0.5) / (total_bad + 0.5))
        )
        nearest_woe = min(bin_woe_ordered, key=lambda w: abs(w - missing_woe))
        frozen[var] = {'type': 'numeric', 'edges': edges, 'bin_woe_ordered': bin_woe_ordered,
                        'zero_woe': None, 'missing_woe': nearest_woe}

    elif var in SINGLE_MERGE_VARS:
        raw_col = RAW_SOURCE[var]
        raw_series = X_train[var]
        merged_series = raw_series.astype('object').fillna('Missing').replace(SINGLE_MERGE_VARS[var])
        category_map = dict(zip(raw_series.astype('object').fillna('Missing'), merged_series))
        temp_df = pd.DataFrame({var: merged_series, 'TARGET': y_train})
        result, iv = calculate_woe_iv_categorical(temp_df, var, min_bad_per_bin=0)
        frozen[var] = {'type': 'categorical', 'table': result, 'category_map': category_map}

    elif var in ['ORGANIZATION_TYPE_BINNED', 'OCCUPATION_TYPE_BINNED']:
        raw_col = RAW_SOURCE[var]
        raw_series = X_train[var]
        collapsed_series = collapse_rare_categories(train_xy.assign(**{raw_col: raw_series}), raw_col)
        category_map = dict(zip(raw_series.astype('object').fillna('Missing'), collapsed_series))
        temp_df = pd.DataFrame({var: collapsed_series, 'TARGET': y_train})
        result, iv = calculate_woe_iv_categorical(temp_df, var, min_bad_per_bin=0)
        frozen[var] = {'type': 'categorical', 'table': result, 'category_map': category_map}

    elif pd.api.types.is_numeric_dtype(X_train[var]):
        result, iv = calculate_woe_iv(train_xy, var, min_bad_per_bin=0)
        edges = extract_edges(result)
        bin_woe_ordered = ordered_bin_woe(result)
        missing_rows = result[result['bin'] == 'Missing']
        frozen[var] = {'type': 'numeric', 'edges': edges, 'bin_woe_ordered': bin_woe_ordered,
                        'zero_woe': None,
                        'missing_woe': missing_rows['woe'].values[0] if len(missing_rows) else None}

    else:
        result, iv = calculate_woe_iv_categorical(train_xy, var, min_bad_per_bin=0)
        frozen[var] = {'type': 'categorical', 'table': result, 'category_map': None}

print(f"Frozen structures built for {len(frozen)} of {len(FINAL_17_VARS)} variables.")

AVISO [EXT_SOURCE_2]: 1 bin(s) com menos de 100 casos 'bad', WoE pode ser instável:
    bin  n_bad
Missing     41

'EXT_SOURCE_2': Missing (WoE=0.0375) fundido com bin (0.512, 0.566] (WoE original=0.0847)
AVISO [DAYS_LAST_PHONE_CHANGE]: 1 bin(s) com menos de 100 casos 'bad', WoE pode ser instável:
    bin  n_bad
Missing      0

'DAYS_LAST_PHONE_CHANGE': Missing (WoE=-1.3336) fundido com bin (-363.0, -161.0] (WoE original=-0.2264)
Frozen structures built for 17 of 17 variables.


In [14]:
# Transform X_train and X_val into WoE space using only the frozen
# structures above, pure lookup, nothing recalculated from X_val.

X_train_woe = pd.DataFrame(index=X_train.index)
X_val_woe = pd.DataFrame(index=X_val.index)

for var in FINAL_17_VARS:
    X_train_woe[var] = apply_frozen_woe(X_train[var], frozen[var])
    X_val_woe[var] = apply_frozen_woe(X_val[var], frozen[var])

print("X_train_woe:", X_train_woe.shape, "| X_val_woe:", X_val_woe.shape)
print("Nulls (train):", X_train_woe.isnull().sum().sum(), "| Nulls (val):", X_val_woe.isnull().sum().sum())

X_train_woe: (246008, 17) | X_val_woe: (61503, 17)
Nulls (train): 0 | Nulls (val): 0


In [15]:
# Final validation: does the rebuilt pipeline reproduce the numbers already
# published in the README and the technical presentation? This is the real
# test, not just "the code runs without error".

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve

# ... (célula de congelamento das 17 variáveis, usando as funções copiadas do 04)

model_check = LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced')
model_check.fit(X_train_woe, y_train)
auc_check = roc_auc_score(y_val, model_check.predict_proba(X_val_woe)[:, 1])

print("AUC reproduzido:", round(auc_check, 4), "| Esperado: 0.7406")
print("Bate?", abs(auc_check - 0.7406) < 0.0005)

AUC reproduzido: 0.7405 | Esperado: 0.7406
Bate? True


In [16]:
# Extract coefficients and intercept from model_check, then recompute
# Factor/Offset using the exact calibration already validated in 04:
# PDO=40, Score_ref=500, Offset anchored on the model's own empirical
# mean log-odds on training data (not a theoretical bad-rate-implied value,
# since class_weight='balanced' shifts the intercept away from the true prior).

PDO = 40
SCORE_REF = 500

factor = PDO / np.log(2)
avg_log_odds_train = model_check.decision_function(X_train_woe).mean()
offset = SCORE_REF - factor * (-avg_log_odds_train)

intercept = model_check.intercept_[0]
coefficients = dict(zip(FINAL_17_VARS, model_check.coef_[0]))

n_characteristics = len(FINAL_17_VARS)
base_points_per_char = (offset - factor * (-intercept)) / n_characteristics

print("Factor:", round(factor, 4))
print("Offset:", round(offset, 4))
print("Base points per characteristic:", round(base_points_per_char, 4))
print()
print("Coeficientes:")
for var, coef in sorted(coefficients.items(), key=lambda x: x[1]):
    print(f"  {var}: {coef:.4f}")

Factor: 57.7078
Offset: 480.515
Base points per characteristic: 28.2661

Coeficientes:
  EXT_SOURCE_2: -0.7634
  EXT_SOURCE_3: -0.7584
  NAME_EDUCATION_TYPE_BINNED: -0.5877
  EXT_SOURCE_1: -0.5273
  AMT_CREDIT: -0.5205
  CODE_GENDER: -0.4669
  YEARS_EMPLOYED: -0.3606
  ORGANIZATION_TYPE_BINNED: -0.3495
  FLOORSMAX_AVG: -0.3235
  BUREAU_CREDIT_ACTIVE_ACTIVE_COUNT: -0.2763
  OCCUPATION_TYPE_BINNED: -0.2392
  REGION_POPULATION_RELATIVE: -0.2033
  DAYS_ID_PUBLISH: -0.1964
  BUREAU_AMT_CREDIT_SUM_LIMIT_MEAN: -0.1763
  BUREAU_AMT_CREDIT_SUM_DEBT_MEAN: -0.1714
  DAYS_LAST_PHONE_CHANGE: -0.1628
  BUREAU_DAYS_CREDIT_MEAN: -0.0792


In [17]:
# Sanity check: pick one validation client, sum their points across all 17
# variables (base + WoE-driven contribution), and confirm it matches the
# score computed directly from the model's log-odds.

def compute_points(raw_value, spec, coef, factor, base_points):
    if spec['type'] == 'numeric':
        bin_idx = pd.cut(pd.Series([raw_value]), bins=spec['edges'], labels=False).iloc[0]
        if pd.isnull(raw_value):
            woe = spec['missing_woe']
        elif spec.get('zero_woe') is not None and raw_value == 0:
            woe = spec['zero_woe']
        else:
            woe = spec['bin_woe_ordered'][int(bin_idx)]
    else:
        table = spec['table']
        category_map = spec['category_map']
        v = raw_value if pd.notnull(raw_value) else 'Missing'
        if category_map is not None:
            fallback = 'Other_grouped' if 'Other_grouped' in table['bin'].values else 'Missing'
            v = category_map.get(v, fallback)
        woe = dict(zip(table['bin'], table['woe']))[v]
    return base_points + (-factor * coef * woe), woe

sample_idx = X_val.index[0]
total_points = 0
for var in FINAL_17_VARS:
    raw_value = X_val.loc[sample_idx, var]
    points, woe = compute_points(raw_value, frozen[var], coefficients[var], factor, base_points_per_char)
    total_points += points

direct_score = offset + factor * (-model_check.decision_function(X_val_woe.loc[[sample_idx]])[0])

print("Soma dos pontos por característica:", round(total_points, 2))
print("Score calculado diretamente:", round(direct_score, 2))

Soma dos pontos por característica: 486.21
Score calculado diretamente: 486.19


In [18]:
# Serialize the frozen scorecard pipeline into a single, portable JSON
# artifact. Infinity is replaced by a large sentinel (+-1e18) for strict
# JSON compatibility across languages, not just Python's own json module.

import json
from datetime import date
from sklearn.metrics import roc_curve

INF_SENTINEL = 1e18

def clean_edges(edges):
    return [(-INF_SENTINEL if e == -np.inf else INF_SENTINEL if e == np.inf else e) for e in edges]

# Recompute KS/Gini fresh, on this reconstructed pipeline
y_pred_proba = model_check.predict_proba(X_val_woe)[:, 1]
fpr, tpr, _ = roc_curve(y_val, model_check.decision_function(X_val_woe))
ks_stat = max(tpr - fpr)
gini = 2 * auc_check - 1

pipeline_artifact = {
    "metadata": {
        "model_name": "home_credit_scorecard",
        "version": "1.0",
        "created": str(date.today()),
        "n_variables": len(FINAL_17_VARS),
        "auc_validation": round(auc_check, 4),
        "ks_validation": round(ks_stat, 4),
        "gini_validation": round(gini, 4),
        "notes": "Rebuilt and validated in 06_model_packaging.ipynb; reproduces 04_scorecard_woe.ipynb within rounding."
    },
    "factor": factor,
    "offset": offset,
    "intercept": intercept,
    "base_points_per_characteristic": base_points_per_char,
    "variables": {}
}

for var in FINAL_17_VARS:
    spec = frozen[var]
    coef = coefficients[var]
    entry = {"type": spec["type"], "coefficient": coef}

    if spec["type"] == "numeric":
        entry["edges"] = clean_edges(spec["edges"])
        entry["bin_woe"] = spec["bin_woe_ordered"]
        entry["zero_woe"] = spec.get("zero_woe")
        entry["missing_woe"] = spec.get("missing_woe")
    else:
        table = spec["table"]
        entry["category_woe"] = dict(zip(table["bin"], table["woe"]))
        entry["category_map"] = spec["category_map"]

    pipeline_artifact["variables"][var] = entry

with open("../models/scorecard_pipeline.json", "w") as f:
    json.dump(pipeline_artifact, f, indent=2)

print("Saved to ../models/scorecard_pipeline.json")
print("AUC:", pipeline_artifact["metadata"]["auc_validation"],
      "| KS:", pipeline_artifact["metadata"]["ks_validation"],
      "| Gini:", pipeline_artifact["metadata"]["gini_validation"])

Saved to ../models/scorecard_pipeline.json
AUC: 0.7405 | KS: 0.3611 | Gini: 0.4811


In [19]:
# Load the saved JSON fresh, from disk only, with no dependency on any
# variable still in memory from this notebook session. This is the real
# test of whether the artifact is production-ready.

with open("../models/scorecard_pipeline.json") as f:
    loaded_pipeline = json.load(f)

print("Metadata:", loaded_pipeline["metadata"])
print()
print("Variáveis salvas:", list(loaded_pipeline["variables"].keys()))
print()
print("Exemplo, EXT_SOURCE_2:")
print("  Coeficiente:", loaded_pipeline["variables"]["EXT_SOURCE_2"]["coefficient"])
print("  Missing WoE:", loaded_pipeline["variables"]["EXT_SOURCE_2"]["missing_woe"])
print("  Número de bins:", len(loaded_pipeline["variables"]["EXT_SOURCE_2"]["bin_woe"]))

Metadata: {'model_name': 'home_credit_scorecard', 'version': '1.0', 'created': '2026-09-08', 'n_variables': 17, 'auc_validation': 0.7405, 'ks_validation': 0.3611, 'gini_validation': 0.4811, 'notes': 'Rebuilt and validated in 06_model_packaging.ipynb; reproduces 04_scorecard_woe.ipynb within rounding.'}

Variáveis salvas: ['EXT_SOURCE_2', 'EXT_SOURCE_3', 'NAME_EDUCATION_TYPE_BINNED', 'EXT_SOURCE_1', 'AMT_CREDIT', 'CODE_GENDER', 'YEARS_EMPLOYED', 'ORGANIZATION_TYPE_BINNED', 'FLOORSMAX_AVG', 'BUREAU_CREDIT_ACTIVE_ACTIVE_COUNT', 'BUREAU_AMT_CREDIT_SUM_LIMIT_MEAN', 'OCCUPATION_TYPE_BINNED', 'REGION_POPULATION_RELATIVE', 'DAYS_ID_PUBLISH', 'BUREAU_AMT_CREDIT_SUM_DEBT_MEAN', 'DAYS_LAST_PHONE_CHANGE', 'BUREAU_DAYS_CREDIT_MEAN']

Exemplo, EXT_SOURCE_2:
  Coeficiente: -0.7633965270029892
  Missing WoE: 0.0839242615872386
  Número de bins: 10


In [20]:
# score_client: the function that will be called directly by the future API.
# Takes a dict of raw feature values for one client and the loaded pipeline
# dict, returns the score, PD estimate, and per-variable point breakdown.
# Depends only on the pipeline dict (loaded from disk) and pandas/numpy,
# never on any notebook-session state (frozen, model_check, etc.).

def score_client(raw_features: dict, pipeline: dict) -> dict:
    factor = pipeline["factor"]
    base_points = pipeline["base_points_per_characteristic"]

    breakdown = {}
    total_points = 0.0

    for var, spec in pipeline["variables"].items():
        raw_value = raw_features.get(var)
        coef = spec["coefficient"]

        if spec["type"] == "numeric":
            if raw_value is None or (isinstance(raw_value, float) and pd.isnull(raw_value)):
                woe = spec["missing_woe"]
            elif spec.get("zero_woe") is not None and raw_value == 0:
                woe = spec["zero_woe"]
            else:
                bin_idx = pd.cut([raw_value], bins=spec["edges"], labels=False)[0]
                woe = spec["bin_woe"][int(bin_idx)]
        else:
            v = raw_value if raw_value is not None else "Missing"
            category_map = spec.get("category_map")
            if category_map is not None:
                fallback = "Other_grouped" if "Other_grouped" in spec["category_woe"] else "Missing"
                v = category_map.get(v, fallback)
            woe = spec["category_woe"].get(v, spec["category_woe"].get("Missing", 0.0))

        points = base_points + (-factor * coef * woe)
        breakdown[var] = {"raw_value": raw_value, "woe": round(woe, 4), "points": round(points, 2)}
        total_points += points

    intercept = pipeline["intercept"]
    log_odds = sum(spec["coefficient"] * breakdown[var]["woe"] for var, spec in pipeline["variables"].items()) + intercept
    probability_default = 1 / (1 + np.exp(-log_odds))

    return {
        "score": round(total_points, 2),
        "probability_default": round(probability_default, 4),
        "breakdown": breakdown,
    }

In [24]:
# Find the row where SK_ID_CURR equals 251294 (not a positional index).
sample_client_idx = X_val[X_val['SK_ID_CURR'] == 251294].index

if len(sample_client_idx) == 0:
    print("Cliente 251294 não está no conjunto de validação (pode estar no treino, já que o split é aleatório).")
else:
    sample_client_idx = sample_client_idx[0]
    raw_client = X_val.loc[sample_client_idx, FINAL_17_VARS].to_dict()

    result = score_client(raw_client, loaded_pipeline)
    print("Score:", result["score"])
    print("Probabilidade de default:", result["probability_default"])
    print()
    for var, info in sorted(result["breakdown"].items(), key=lambda x: x[1]["points"]):
        print(f"  {var}: raw={info['raw_value']}, woe={info['woe']}, points={info['points']}")

Score: 670.81
Probabilidade de default: 0.0357

  ORGANIZATION_TYPE_BINNED: raw=Business Entity Type 3, woe=-0.1512, points=25.22
  BUREAU_AMT_CREDIT_SUM_LIMIT_MEAN: raw=nan, woe=-0.1741, points=26.49
  DAYS_LAST_PHONE_CHANGE: raw=-962.0, woe=-0.0315, points=27.97
  BUREAU_DAYS_CREDIT_MEAN: raw=-952.2, woe=0.1284, points=28.85
  BUREAU_CREDIT_ACTIVE_ACTIVE_COUNT: raw=2.0, woe=0.0877, points=29.66
  DAYS_ID_PUBLISH: raw=-5545, woe=0.3193, points=31.89
  OCCUPATION_TYPE_BINNED: raw=Managers, woe=0.28, points=32.13
  BUREAU_AMT_CREDIT_SUM_DEBT_MEAN: raw=0.0, woe=0.4079, points=32.3
  CODE_GENDER: raw=F, woe=0.1541, points=32.42
  REGION_POPULATION_RELATIVE: raw=0.04622, woe=0.6063, points=35.38
  FLOORSMAX_AVG: raw=0.3333, woe=0.4164, points=36.04
  AMT_CREDIT: raw=1223010.0, woe=0.4325, points=41.26
  YEARS_EMPLOYED: raw=15.2, woe=0.6312, points=41.4
  NAME_EDUCATION_TYPE_BINNED: raw=Higher education, woe=0.4333, points=42.96
  EXT_SOURCE_1: raw=0.8683837407692271, woe=1.1978, points=64.